# Predispatch Source

In [3]:
from __future__ import annotations
import subprocess
import traceback
from pathlib import Path
import pandas as pd
import os
import textwrap
import urllib.request, zipfile, io

Load predispatch step-horizon price features.

In [4]:

def _fetch_predispatch(run_start_str: str, run_end_str: str, forecasted_end_str: str):
    month = pd.Timestamp(run_start_str)

    # if month >= pd.Timestamp("2024/08/01"):
    #     # Aug 2024+: AEMO renamed archive files; nemseer can't handle them, so download manually and call clean_forecast_csv() directly.
    #     cache_path = Path("Pre_processing/nemseer_predispatch/extracted")
    #     year, m = month.year, str(month.month).zfill(2)
    #     old_name   = f"PUBLIC_DVD_PREDISPATCHPRICE_{year}{m}010000"
    #     csv_path     = cache_path / f"{old_name}.CSV"
    #     parquet_path = cache_path / f"{old_name}.parquet"

    #     if not csv_path.exists() and not parquet_path.exists():
    #         new_file = f"PUBLIC_ARCHIVE%23PREDISPATCHPRICE%23ALL%23FILE01%23{year}{m}010000.zip"
    #         url = (
    #             f"http://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/"
    #             f"{year}/MMSDM_{year}_{m}/MMSDM_Historical_Data_SQLLoader/PREDISP_ALL_DATA/{new_file}"
    #         )
    #         print(f"  Downloading new-format archive for {year}-{m}...", flush=True)
    #         req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    #         with urllib.request.urlopen(req) as resp:
    #             zip_data = resp.read()
    #         z = zipfile.ZipFile(io.BytesIO(zip_data))
    #         csvfiles = [n for n in z.namelist() if n.upper().endswith(".CSV")]
    #         if not csvfiles:
    #             raise RuntimeError(f"No CSV found in new-format archive for {year}-{m}")
    #         with open(csv_path, "wb") as f:
    #             f.write(z.read(csvfiles[0]))

        # Subprocess: use nemseer's CSV parser directly (bypass download/validation)
        code = textwrap.dedent(f"""
            import pandas as pd
            from pathlib import Path
            from nemseer.data_handlers import clean_forecast_csv

            parquet_path = Path({str(parquet_path)!r})
            csv_path     = Path({str(csv_path)!r})

            if parquet_path.exists():
                df = pd.read_parquet(parquet_path)
            elif csv_path.exists():
                df = clean_forecast_csv(csv_path)
                df.to_parquet(parquet_path, index=False)
            else:
                raise FileNotFoundError(f"Expected CSV not found: {{csv_path}}")

            df.to_parquet({"Pre_processing/nemseer_predispatch/temp.parquet"!r}, index=False)
        """)

    else:
        # --- Old-format path (before Aug 2024): use nemseer normally ---
        code = textwrap.dedent(f"""
            import nemseer, pandas as pd
            data = nemseer.compile_data(
                run_start={run_start_str!r},
                run_end={run_end_str!r},
                forecasted_start={run_start_str!r},
                forecasted_end={forecasted_end_str!r},
                raw_cache="Pre_processing/nemseer_predispatch/extracted",
                processed_cache="Pre_processing/nemseer_predispatch/processed",
                forecast_type="PREDISPATCH",
                tables=["PRICE"],
            )
            if isinstance(data, dict):
                df = data.get("PRICE")
                if df is None:
                    df = data.get("price")
            else:
                df = data.to_dataframe().reset_index()
            df.to_parquet({"Pre_processing/nemseer_predispatch/temp.parquet"!r}, index=False)
        """)

    # Run in the nemseer Python environment and return the output
    result = subprocess.run(
        [r"C:\Users\danie\conda_envs\nemseer_env\python.exe", "-c", code],
        capture_output=True, text=True, cwd=Path.cwd(),
    )

    # if result.returncode != 0:
    #     raise RuntimeError(
    #         f"nemseer subprocess failed for {run_start_str[:7]} (exit {result.returncode}):\n"
    #         f"STDOUT:\n{result.stdout if result.stdout else '(none)'}\n"
    #         f"STDERR:\n{result.stderr if result.stderr else '(none)'}"
    #     )

    # df = pd.read_parquet("Pre_processing/nemseer_predispatch/temp.parquet")
    # os.remove("Pre_processing/nemseer_predispatch/temp.parquet")
    return df


# def _cleanup_month_cache(month: pd.Timestamp) -> None:
#     """Delete nemseer cache files for the given month from both extracted and processed dirs."""
#     pattern = month.strftime("%Y%m")
#     for cache_dir in [
#         Path("Pre_processing/nemseer_predispatch/extracted"),
#         Path("Pre_processing/nemseer_predispatch/processed"),
#     ]:
#         if cache_dir.exists():
#             for f in cache_dir.glob(f"*{pattern}*"):
#                 if f.is_file():
#                     f.unlink()


# def main():
    # data_start  = pd.Timestamp("2018/01/01")
    # data_end    = pd.Timestamp("2018/02/01")

    # Go back 1 day so the previous day's overnight runs (which forecast into data_start) are included
    # fetch_start = data_start - pd.Timedelta(days=1)
    # fetch_start_month = fetch_start.to_period("M").to_timestamp()
    # total_months = pd.date_range(fetch_start_month, data_end - pd.Timedelta(days=1), freq="MS")

    # Determine which months have already been processed
    # processed_path = Path("Processed_data/6_predispatch.csv")
    # already_processed = set()
    # if processed_path.exists():
    #     existing_idx = pd.read_csv(processed_path, usecols=[0], parse_dates=True).iloc[:, 0]
    #     already_processed = set(pd.to_datetime(existing_idx).dt.to_period("M"))
    #     print(f"  Found {len(already_processed)} already-processed month(s) — will skip.", flush=True)

    # monthly_slim = []

    # for i, month in enumerate(total_months):
    #     # For the Dec 2017 pre-fetch month, use Jan 2018 as the proxy period to check
    #     check_period = max(month, data_start).to_period("M")
    #     if check_period in already_processed:
    #         print(f"{i + 1:3d}/{len(total_months)} {month:%Y-%m} skipping (already processed).", flush=True)
    #         continue

    #     print(f"{i + 1:3d}/{len(total_months)} {month:%Y-%m} fetching...", flush=True)

        # next_month_4am = (month + pd.offsets.MonthEnd(0) + pd.Timedelta(days=1)).replace(hour=4, minute=0)
        # month_end = next_month_4am if next_month_4am < data_end else data_end - pd.Timedelta(minutes=30)

        # if month < pd.Timestamp("2024/08/01") and month_end >= pd.Timestamp("2024/08/01"):
        #     month_end = pd.Timestamp("2024/08/01") - pd.Timedelta(minutes=30)

        # run_start_str = month.strftime("%Y/%m/%d %H:%M")
        # run_end_str = month_end.strftime("%Y/%m/%d %H:%M")

        # days_ahead = 2 if month_end.hour >= 13 else 1
        # forecasted_end_str = (month_end + pd.Timedelta(days=days_ahead)).normalize() + pd.Timedelta(hours=4)
        # forecasted_end_str = forecasted_end_str.strftime("%Y/%m/%d %H:%M")

        # try:
        #     raw = _fetch_predispatch(run_start_str, run_end_str, forecasted_end_str)
        # except RuntimeError as e:
        #     if not ("404" in str(e) or "Table(s) not available" in str(e)) or month >= pd.Timestamp("2024/08/01"):
        #         raise
            # First-pass failure: the next month's archive boundary runs are missing (e.g.
            # Sep 2022 run_end=Oct 1 04:00 but Oct 2022 ALL_DATA doesn't exist). Retry
            # capped within this month only.
            # month_end_capped = (month + pd.offsets.MonthEnd(0)).replace(hour=23, minute=30)
            # run_end_capped = month_end_capped.strftime("%Y/%m/%d %H:%M")
            # days_ahead_capped = 2 if month_end_capped.hour >= 13 else 1
            # forecasted_end_capped = (
            #     (month_end_capped + pd.Timedelta(days=days_ahead_capped)).normalize()
            #     + pd.Timedelta(hours=4)
            # ).strftime("%Y/%m/%d %H:%M")
            # try:
            #     print(f"  WARNING: next-month archive gap, retrying within {month:%Y-%m} only...", flush=True)
            #     raw = _fetch_predispatch(run_start_str, run_end_capped, forecasted_end_capped)
            # except RuntimeError as e2:
            #     if "404" in str(e2) or "Table(s) not available" in str(e2):
            #         # The ALL_DATA archive is missing for this month (e.g. Oct 2022).
            #         # Fall back to the _D (delta) file — recovers h=1 only.
            #         print(f"  WARNING: ALL_DATA archive missing for {month:%Y-%m}, trying _D delta file...", flush=True)
            #         year, m = month.year, str(month.month).zfill(2)
            #         fname = f"PUBLIC_DVD_PREDISPATCHPRICE_D_{year}{m}010000.zip"
            #         url = (
            #             f"http://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/"
            #             f"{year}/MMSDM_{year}_{m}/MMSDM_Historical_Data_SQLLoader/DATA/{fname}"
            #         )
            #         delta_slim = None
            #         try:
            #             req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            #             with urllib.request.urlopen(req) as resp:
            #                 delta_bytes = resp.read()
            #             z = zipfile.ZipFile(io.BytesIO(delta_bytes))
            #             with z.open(z.namelist()[0]) as fh:
            #                 lines = fh.read().decode("utf-8", errors="replace").splitlines()
            #             data_rows  = [l for l in lines if l.startswith("D,PREDISPATCH,REGION_PRICES")]
            #             header_row = next((l for l in lines if l.startswith("I,PREDISPATCH,REGION_PRICES")), None)
            #             if header_row and data_rows:
            #                 all_cols = header_row.split(",")[4:]
            #                 drows = [r.split(",")[4:] for r in data_rows]
            #                 drows = [r[:len(all_cols)] + [''] * (len(all_cols) - len(r)) for r in drows]
            #                 df_d = pd.DataFrame(drows, columns=[c.strip().upper() for c in all_cols])
            #                 seqno      = df_d["PREDISPATCHSEQNO"].str.strip()
            #                 base_date  = pd.to_datetime(seqno.str[:8], format="%Y%m%d", errors="coerce")
            #                 period_num = pd.to_numeric(seqno.str[8:], errors="coerce")
            #                 run_time_d = base_date + pd.to_timedelta(4 * 60 + 30 * (period_num - 1), unit="min")
            #                 forecast_time_d = pd.to_datetime(df_d["DATETIME"].str.strip('"'), errors="coerce")
            #                 rrp_d           = pd.to_numeric(df_d["RRP"], errors="coerce")
            #                 horizon_step_d  = ((forecast_time_d - run_time_d).dt.total_seconds() / 1800.0).round()
            #                 region_d        = df_d["REGIONID"].str.lower().str.strip()
            #                 mask = horizon_step_d.notna() & (horizon_step_d > 0)
            #                 h_int = horizon_step_d[mask].astype(int)
            #                 delta_slim = pd.DataFrame({
            #                     "run_time": run_time_d[mask].values,
            #                     "col":      ("predispatch_rrp_h" + h_int.astype(str) + "_" + region_d[mask]).values,
            #                     "rrp":      rrp_d[mask].values,
            #                 }).reset_index(drop=True)
            #         except Exception:
            #             delta_slim = None
            #         if delta_slim is not None and not delta_slim.empty:
            #             print(f"  _D file: recovered {delta_slim['run_time'].nunique()} run times (h=1 only) for {month:%Y-%m}", flush=True)
            #             monthly_slim.append(delta_slim)
            #             _cleanup_month_cache(month)
            #         else:
            #             print(f"  No _D file available either, skipping {month:%Y-%m}.", flush=True)
            #         continue
            #     raise

        # Slim down to (run_time, col, rrp) immediately — holding full raw nemseer
        # DataFrames for all months simultaneously would exhaust RAM on concat.
        # raw.columns = [str(c).upper() for c in raw.columns]
        # run_time      = pd.to_datetime(raw["PREDISPATCH_RUN_DATETIME"], errors="coerce")
        # forecast_time = pd.to_datetime(raw["DATETIME"], errors="coerce")
        # rrp           = pd.to_numeric(raw["RRP"], errors="coerce")
        # horizon_step  = ((forecast_time - run_time).dt.total_seconds() / 1800.0).round().astype(int)
        # region        = raw["REGIONID"].str.lower().str.strip()
        # col           = "predispatch_rrp_h" + horizon_step.astype(str) + "_" + region
        # slim          = pd.DataFrame({"run_time": run_time, "col": col, "rrp": rrp})
        # monthly_slim.append(slim[horizon_step > 0].reset_index(drop=True))
        # _cleanup_month_cache(month)
        # print(f"  {month:%Y-%m} cache deleted.", flush=True)

    # if not monthly_slim:
    #     print("No new months to process.", flush=True)
    #     return pd.read_csv(processed_path, index_col=0, parse_dates=True).tail()

    # raw_all = pd.concat(monthly_slim, ignore_index=True)
    # df = raw_all.pivot_table(index="run_time", columns="col", values="rrp", aggfunc="first")
    # df.columns.name = None

    # # Reindex to a complete 30-minute range, leaving gaps as NaN
    # full_range = pd.date_range(data_start + pd.Timedelta(minutes=30), data_end - pd.Timedelta(minutes=30), freq="30min")
    # df = df[(df.index >= fetch_start) & (df.index < data_end)].sort_index()
    # df = df.reindex(full_range)
    # df.index.name = "SETTLEMENTDATE"

    # # Sort columns: grouped by region, ordered by horizon step within each region
    # df = df[sorted(df.columns, key=lambda c: (c.rsplit("_", 1)[-1], int(c.split("_h")[1].split("_")[0])))]

    # if processed_path.exists():
    #     existing_processed = pd.read_csv(processed_path, index_col=0, parse_dates=True)
    #     existing_processed.index = pd.to_datetime(existing_processed.index, format="mixed")
    #     df = pd.concat([existing_processed, df])
    #     df = df[~df.index.duplicated(keep="last")].sort_index()

    # df.to_csv(processed_path)
    # return df.tail()

# main()


  Found 96 already-processed month(s) — will skip.
  1/2 2017-12 skipping (already processed).
  2/2 2018-01 skipping (already processed).
No new months to process.


,predispatch_rrp_h1_nsw1,predispatch_rrp_h2_nsw1,predispatch_rrp_h3_nsw1,predispatch_rrp_h4_nsw1,predispatch_rrp_h5_nsw1,predispatch_rrp_h6_nsw1,predispatch_rrp_h7_nsw1,predispatch_rrp_h8_nsw1,predispatch_rrp_h9_nsw1,predispatch_rrp_h10_nsw1,...,predispatch_rrp_h69_vic1,predispatch_rrp_h70_vic1,predispatch_rrp_h71_vic1,predispatch_rrp_h72_vic1,predispatch_rrp_h73_vic1,predispatch_rrp_h74_vic1,predispatch_rrp_h75_vic1,predispatch_rrp_h76_vic1,predispatch_rrp_h77_vic1,predispatch_rrp_h78_vic1
SETTLEMENTDATE,,,,,,,,,,,,,,,,,,,,,
2025-12-31 21:30:00,83.51410,105.79000,92.26774,57.30000,74.75819,57.30000,70.65651,57.06014,57.06014,57.06014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-31 22:00:00,80.02000,88.17883,57.56014,57.30000,57.30000,71.08423,57.06014,57.06014,57.06014,57.06014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-31 22:30:00,83.89026,57.30000,57.30000,57.30000,76.89694,57.06014,57.06014,57.06014,57.06014,57.06014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-31 23:00:00,83.62451,75.59780,57.30000,77.12281,57.06014,57.06014,57.06014,57.06014,57.06014,57.06014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-31 23:30:00,75.63213,57.30000,76.39143,57.06014,57.06014,57.06014,57.06014,57.06014,57.06014,57.06010,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
